# Week 7 – Incremental Data Processing using Delta Lake

## Objective

To load customer data into a Delta table, clean the data, process incremental records, apply a MERGE operation, and validate the final output.

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

print("Libraries imported successfully")

Libraries imported successfully


In [0]:
master_csv_path = "/Volumes/tanvi_week7_databricks/default/week7_data/customer_master.csv"

incremental_csv_path = "/Volumes/tanvi_week7_databricks/default/week7_data/customer_incremental.csv"

In [0]:
master_raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(master_csv_path)
)

print("Master dataset loaded successfully")
print("Master row count:", master_raw_df.count())

display(master_raw_df)

Master dataset loaded successfully
Master row count: 11


customer_id,customer_name,email,city,phone,updated_at
101,Aarav Sharma,aarav@gmail.com,Mumbai,9876500001,2026-07-01
102,Priya Patel,priya@gmail.com,Ahmedabad,9876500002,2026-07-02
103,Rohan Gupta,rohan@gmail.com,Delhi,9876500003,2026-07-03
104,Sneha Reddy,sneha@gmail.com,Hyderabad,9876500004,2026-07-04
105,Vikram Singh,vikram@gmail.com,Jaipur,9876500005,2026-07-05
106,Ananya Iyer,ananya@gmail.com,Chennai,9876500006,2026-07-06
107,Karan Mehta,karan@gmail.com,Pune,9876500007,2026-07-07
108,Divya Nair,divya@gmail.com,null,9876500008,2026-07-08
109,Arjun Rao,arjun@gmail.com,Bengaluru,9876500009,2026-07-09
110,Neha Joshi,neha@gmail.com,Nagpur,9876500010,2026-07-10


In [0]:
master_raw_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- phone: long (nullable = true)
 |-- updated_at: date (nullable = true)



In [0]:
master_raw_df.select([
    F.count(
        F.when(
            F.col(column).isNull() |
            (F.trim(F.col(column).cast("String")) == ""),
            column
        )
    ).alias(column)
    for column in master_raw_df.columns
]).show()

+-----------+-------------+-----+----+-----+----------+
|customer_id|customer_name|email|city|phone|updated_at|
+-----------+-------------+-----+----+-----+----------+
|          0|            0|    0|   1|    0|         0|
+-----------+-------------+-----+----+-----+----------+



In [0]:
master_duplicate_count = (
    master_raw_df.count()
    - master_raw_df.dropDuplicates(["customer_id"]).count()
)

print("Duplicate customer IDs:", master_duplicate_count)

Duplicate customer IDs: 1


In [0]:
master_clean_df = (
    master_raw_df
    .filter(F.col("customer_id").isNotNull())
    .withColumn(
        "city",
        F.when(
            F.col("city").isNull() |
            (F.trim(F.col("city")) == ""),
            "Unknown"
        ).otherwise(F.trim(F.col("city")))
    )
    .withColumn("customer_name", F.trim(F.col("customer_name")))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .dropDuplicates(["customer_id"])
)

print("Raw rows:", master_raw_df.count())
print("Clean rows:", master_clean_df.count())

display(master_clean_df.orderBy("customer_id"))

Raw rows: 11
Clean rows: 10


customer_id,customer_name,email,city,phone,updated_at
101,Aarav Sharma,aarav@gmail.com,Mumbai,9876500001,2026-07-01
102,Priya Patel,priya@gmail.com,Ahmedabad,9876500002,2026-07-02
103,Rohan Gupta,rohan@gmail.com,Delhi,9876500003,2026-07-03
104,Sneha Reddy,sneha@gmail.com,Hyderabad,9876500004,2026-07-04
105,Vikram Singh,vikram@gmail.com,Jaipur,9876500005,2026-07-05
106,Ananya Iyer,ananya@gmail.com,Chennai,9876500006,2026-07-06
107,Karan Mehta,karan@gmail.com,Pune,9876500007,2026-07-07
108,Divya Nair,divya@gmail.com,Unknown,9876500008,2026-07-08
109,Arjun Rao,arjun@gmail.com,Bengaluru,9876500009,2026-07-09
110,Neha Joshi,neha@gmail.com,Nagpur,9876500010,2026-07-10


In [0]:
delta_table_path = "/Volumes/tanvi_week7_databricks/default/week7_data/customer_delta"

master_clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_table_path)

print("Delta table created successfully")

Delta table created successfully


In [0]:
delta_df = spark.read.format("delta").load(delta_table_path)

display(delta_df)

customer_id,customer_name,email,city,phone,updated_at
105,Vikram Singh,vikram@gmail.com,Jaipur,9876500005,2026-07-05
104,Sneha Reddy,sneha@gmail.com,Hyderabad,9876500004,2026-07-04
108,Divya Nair,divya@gmail.com,Unknown,9876500008,2026-07-08
109,Arjun Rao,arjun@gmail.com,Bengaluru,9876500009,2026-07-09
106,Ananya Iyer,ananya@gmail.com,Chennai,9876500006,2026-07-06
103,Rohan Gupta,rohan@gmail.com,Delhi,9876500003,2026-07-03
107,Karan Mehta,karan@gmail.com,Pune,9876500007,2026-07-07
102,Priya Patel,priya@gmail.com,Ahmedabad,9876500002,2026-07-02
101,Aarav Sharma,aarav@gmail.com,Mumbai,9876500001,2026-07-01
110,Neha Joshi,neha@gmail.com,Nagpur,9876500010,2026-07-10


In [0]:
incremental_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(incremental_csv_path)
)

display(incremental_df)


customer_id,customer_name,email,city,phone,updated_at
102,Priya Patel,priya.patel@gmail.com,Pune,9876500002,2026-08-01
104,Sneha Reddy,sneha.reddy@gmail.com,Bengaluru,9876500004,2026-08-01
107,Karan Mehta,karan@gmail.com,Mumbai,9876500007,2026-08-01
111,Riya Kapoor,riya@gmail.com,Pune,9876500011,2026-08-01
112,Aditya Kulkarni,aditya@gmail.com,null,9876500012,2026-08-01
113,Meera Shah,meera@gmail.com,Surat,9876500013,2026-08-01
113,Meera Shah,meera@gmail.com,Surat,9876500013,2026-08-01


In [0]:
incremental_clean_df = (
    incremental_df
    .filter(F.col("customer_id").isNotNull())
    .withColumn(
        "city",
        F.when(
            F.col("city").isNull() |
            (F.trim(F.col("city")) == ""),
            "Unknown"
        ).otherwise(F.trim(F.col("city")))
    )
    .withColumn("customer_name", F.trim(F.col("customer_name")))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .dropDuplicates(["customer_id"])
)

print("Raw incremental rows:", incremental_df.count())
print("Clean incremental rows:", incremental_clean_df.count())

display(incremental_clean_df.orderBy("customer_id"))

Raw incremental rows: 7
Clean incremental rows: 6


customer_id,customer_name,email,city,phone,updated_at
102,Priya Patel,priya.patel@gmail.com,Pune,9876500002,2026-08-01
104,Sneha Reddy,sneha.reddy@gmail.com,Bengaluru,9876500004,2026-08-01
107,Karan Mehta,karan@gmail.com,Mumbai,9876500007,2026-08-01
111,Riya Kapoor,riya@gmail.com,Pune,9876500011,2026-08-01
112,Aditya Kulkarni,aditya@gmail.com,Unknown,9876500012,2026-08-01
113,Meera Shah,meera@gmail.com,Surat,9876500013,2026-08-01


In [0]:
deltaTable = DeltaTable.forPath(spark, delta_table_path)

(
    deltaTable.alias("target")
    .merge(
        incremental_clean_df.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("MERGE completed successfully")

MERGE completed successfully


In [0]:
final_df = spark.read.format("delta").load(delta_table_path)

print("Final Row Count:", final_df.count())

display(final_df.orderBy("customer_id"))

Final Row Count: 13


customer_id,customer_name,email,city,phone,updated_at
101,Aarav Sharma,aarav@gmail.com,Mumbai,9876500001,2026-07-01
102,Priya Patel,priya.patel@gmail.com,Pune,9876500002,2026-08-01
103,Rohan Gupta,rohan@gmail.com,Delhi,9876500003,2026-07-03
104,Sneha Reddy,sneha.reddy@gmail.com,Bengaluru,9876500004,2026-08-01
105,Vikram Singh,vikram@gmail.com,Jaipur,9876500005,2026-07-05
106,Ananya Iyer,ananya@gmail.com,Chennai,9876500006,2026-07-06
107,Karan Mehta,karan@gmail.com,Mumbai,9876500007,2026-08-01
108,Divya Nair,divya@gmail.com,Unknown,9876500008,2026-07-08
109,Arjun Rao,arjun@gmail.com,Bengaluru,9876500009,2026-07-09
110,Neha Joshi,neha@gmail.com,Nagpur,9876500010,2026-07-10


In [0]:
duplicate_count = (
    final_df.count()
    - final_df.dropDuplicates(["customer_id"]).count()
)

print("Duplicate Customer IDs:", duplicate_count)

Duplicate Customer IDs: 0


In [0]:
display(final_df.orderBy("customer_id"))

customer_id,customer_name,email,city,phone,updated_at
101,Aarav Sharma,aarav@gmail.com,Mumbai,9876500001,2026-07-01
102,Priya Patel,priya.patel@gmail.com,Pune,9876500002,2026-08-01
103,Rohan Gupta,rohan@gmail.com,Delhi,9876500003,2026-07-03
104,Sneha Reddy,sneha.reddy@gmail.com,Bengaluru,9876500004,2026-08-01
105,Vikram Singh,vikram@gmail.com,Jaipur,9876500005,2026-07-05
106,Ananya Iyer,ananya@gmail.com,Chennai,9876500006,2026-07-06
107,Karan Mehta,karan@gmail.com,Mumbai,9876500007,2026-08-01
108,Divya Nair,divya@gmail.com,Unknown,9876500008,2026-07-08
109,Arjun Rao,arjun@gmail.com,Bengaluru,9876500009,2026-07-09
110,Neha Joshi,neha@gmail.com,Nagpur,9876500010,2026-07-10


In [0]:
print("========== Assignment Summary ==========")
print("✓ Loaded master dataset")
print("✓ Cleaned null values")
print("✓ Removed duplicates")
print("✓ Created Delta Table")
print("✓ Loaded incremental dataset")
print("✓ Applied MERGE (Upsert)")
print("✓ Validated row count")
print("✓ Validated duplicate count")
print("✓ Displayed final dataset")

========== Assignment Summary ==========
✓ Loaded master dataset
✓ Cleaned null values
✓ Removed duplicates
✓ Created Delta Table
✓ Loaded incremental dataset
✓ Applied MERGE (Upsert)
✓ Validated row count
✓ Validated duplicate count
✓ Displayed final dataset
